# 07_requests.ipynb
1. `%uv add -q requests` -> 앞에 %가 붙어있으면 python 셀 이지만, 터미널에 실행할 명령어
2. `pip install requests` -> 우리는 최신기술 uv쓸거라 상황에 따라 이거 쓸 수 도 있음

In [ ]:
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'

res = requests.get(URL)
ram = "성훈"

# 데이터 덩어리를 해석 할 수 있는(ex dict 형태로 바뀌는 등) 형태로 바꾸는 작업이 파싱(parsing) 이라고 불림
type(res.text)   #str -> 단순 글자 상태 -> 파싱 안된 데이터
type(res.json()) #dict -> key,value 값으로 받을 수 있어서(표 형식) 원하는 데이터만 뽑아 낼 수 있다 -> 파싱 된 데이터

raw_data = res.text
parsing_data = res.json()

res.json()

# 1인당 1등 당첨 금액 = 1197258718, key값 : #rnk1WnAmt

print(parsing_data['data']['list'][0]['rnk1WnAmt'])
res.json()

In [ ]:
lt_data = parsing_data['data']['list'][0]
lucky = []
bonus = 0

#딕셔너리 순회도 가능하다
for k,v in lt_data.items():
  if 'tm' in k:
   lucky.append(v)
  elif 'bns' in k:
    bonus = v

print(lucky)
print(bonus)

In [ ]:
# Main Mission 
# 랜덤하게 뽑은 번호 6개와, 실제 당첨번호를 비교하여 몇등인지 출력하는 프로그램
# (추가미션) 함수로 만들기

# 로또 RULE) 
# 1등 숫자 6개 같음
# 2등: 숫자 5개 같고 + 나머지 하나가 보너스 번호
# 3등 ~ 5등 : 숫자 5개, 4개, 3개 같음

import random

# 랜덤 번호 추출 함수
def random_nums():
 return random.sample(range(1,46), 6)

# URL request로 당첨 로또 번호 불러오는 함수  
def print_lt_nums():
 lucky_nums = []#보너스 번호 포함된 로또 번호들

 for k,v in lt_data.items():
   if 'tm' in k:
    lucky_nums.append(v)
   elif 'bns' in k:
     lucky_nums.append(v)

 return lucky_nums

# 보너스 번호 불러 오는 함수
def print_bns_num():
 return lt_data['bnsWnNo']

# 당첨 등수 출력 함수
def check_win(random_nums, lucky_nums, bns_num):
 check_num = 0
 is_bns = False
 #로또 번호 대조
 for random_num in random_nums:
  if random_num in lucky_nums:
   check_num += 1
  if random_num == bns_num:
   is_bns = True
 #등수 출력
 if check_num == 6 and is_bns == False:
  return '1등'
 elif check_num == 6 and is_bns == True:
  return '2등'
 elif check_num >= 5:
  return '3등'
 elif check_num >= 4:
  return '4등'
 elif check_num >= 3:
  return '5등'
 else:
  return '꽝'

print(check_win(random_nums(), print_lt_nums(), print_bns_num()))
print(check_win({2,13,18,32,22}, print_lt_nums(), print_bns_num()))



## API 키 관리
1. `uv add python-dotnev`
2. 모든 키 파일은 `.env` 파일에 보관
3. 소스코드에서는 `load_dotenv()와` `os.getenv()`를 사용하여 불러옴

In [ ]:
import os
from dotenv import load_dotenv
# .env 파일 불러오기
load_dotenv()
# 불러온 파일에서 원하는 key 꺼내기
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')


In [ ]:
#네이버 API 식 요청 방식(인증 관련 정보)
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'

headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

In [ ]:
import requests
# 클래식한 쿼리 파라미터
URL = BASE_URL + NEWS_URL

# 쿼리 파라미터를 dict 로 작성
params = {
  'query': '엔화',
  'sort': 'sim',
  'dispaly' : 5, #기사 100개 모아서
}

print(URL)

res = requests.get(URL, headers=headers, params=params)
res.json()

In [ ]:
# 100개 기사를 모아서
# title에 <b>, </b> 이상한 태그 없애기
# 조건 : link URL이 naver 뉴스인 애들만 모아야 함.(100개가 안 될 수 있습니다).
# 간략히 다음과 같은 모양으로 만들기
# news 변수 내용을 csv 로 export 하기

'''
news = [
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'},
  {'title': '제목제목', 'link': 'https://skdflajsdklf.com'}
]
'''





## Parsing
1. JSON 문자열 -> dict 로 해석
2. HTML 문자열 -> 구조화 필요 (`Beautifulsoup4`)

In [ ]:
# uv add beautifulsoup4
import requests
from bs4 import BeautifulSoup

URL = 'https://n.news.naver.com/article/008/0005405342'

def extract_naver_news(url):
   # 네이버 뉴스 아니면 에러발생
   if 'n.news.naver.com' not in url:
     #예외처리로 에러 발생 시키는 코드
     raise Exception('네이버 뉴스가 아닙니다')

   res = requests.get(url)

   #res.text 를 해석 완료!
   soup = BeautifulSoup(res.text, 'html.parser')

   #해석한 HTML 에서 '#dic_area' 선택자로 추출 -> 글자만 뽑아서 -> 양옆 공백(엔터, 스페이스) 삭제
   news_text = soup.select_one('#dic_area').text.strip()

   return news_text



extract_naver_news(URL)


In [ ]:
# 1. 특정 주제로 Naver News 연관도 순으로 5개 뽑기
# 2. Naver 뉴스 링크를 통해서 본문만 추출하기
# 3. 최종 결과형식

'''
news = [
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'},
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'},
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'},
  {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문', 'comment':'좋아요'}
]
'''